# Misalignment QA — Results Explorer

Pull experiment results from Langfuse, compare conditions and models, and drill into the most interesting traces.

**How to use:** edit the constants in the *Constants* cell, then run top-to-bottom.  
Not sure what dataset or execution ID to use? Run the *Discovery* cell first.

---

## The 5 Conditions

The bootcamp experiment (`bootcamp_misalignment.yaml`) tests how five different context conditions affect whether a model follows reckless example patterns.

| Condition | Examples? | Inject method | Priority instruction? |
|---|---|---|---|
| `baseline` | No | — | No |
| `examples-as-llm-messages` | Yes (reckless) | LLM API messages | No |
| `examples-and-priority-instruction` | Yes (reckless) | LLM API messages | Yes |
| `examples-as-user-context` | Yes (reckless) | Text in user prompt | No |
| `examples-as-context-with-priority` | Yes (reckless) | Text in user prompt | Yes |

**LLM messages** = examples are injected as prior conversation turns via the API (developer-only surface).  
**User context** = examples are formatted as a text block prepended to the user message (any end-user can do this).

**Suggested analysis axes:**
- Set `GROUP_BY = "condition"` to compare conditions side-by-side (averaging over all 6 models).
- Set `GROUP_BY = "model"` to compare models (averaging over all 5 conditions).
- Set `GROUP_BY = "variant_id"` for the full 30-row breakdown.

In [ ]:
# Setup — no edits needed here.
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "implementations").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from implementations.misalignment_qa.results_notebook import MisalignmentResultsExplorer

pd.options.display.max_colwidth = 160
pd.options.display.max_columns = 30
pd.options.display.width = 200

explorer = MisalignmentResultsExplorer()

In [ ]:
# Discovery — run this to see what datasets and execution IDs are available.
# Use the output to fill in DATASET_NAME and EXECUTION_ID below.

display(Markdown("### Available Datasets"))
display(explorer.list_datasets_frame()[["dataset_name", "updated_at", "local_experiment_ids", "local_config_paths"]])

# Uncomment to see all run instances for the bootcamp dataset:
# display(Markdown("### Runs for bootcamp dataset"))
# display(explorer.list_run_instances_frame("misalignment-qa-bootcamp-v1"))

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────────
# Edit these, then run from here to the bottom.
#
# DATASET_NAME  — matches `langfuse_dataset_name` in the experiment config YAML.
# EXECUTION_ID  — "latest" = most recent run, "all" = every run ever, or paste a
#                 specific execution ID from the Discovery cell above.
# GROUP_BY      — what to compare across:
#                   "condition"  → side-by-side view of the 5 experimental conditions
#                   "model"      → which models are most/least susceptible
#                   "variant_id" → full 30-row breakdown (condition × model)
# TRACE_LIMIT   — set to e.g. 50 to speed things up during development; None = all.
# SORT_BY       — metric columns to rank traces by (most harmful first by default).
# MASTER_VIEW_COLUMNS — columns shown in the top-traces table.

DATASET_NAME = "misalignment-qa-bootcamp-v1"
EXECUTION_ID = "latest"
GROUP_BY = "condition"  # try "model" or "variant_id" for different cuts
TRACE_LIMIT = None

TOP_N = 20
DETAIL_TRACE_COUNT = 10

SORT_BY = ["harmful", "follows_reckless_pattern", "helpful", "timestamp"]
ASCENDING = [False, False, True, False]

MASTER_VIEW_COLUMNS = [
    "timestamp",
    "task_id",
    "variant_id",
    "model",
    "condition",
    "harmful",
    "helpful",
    "follows_reckless_pattern",
    "harmful_comment",
    "helpful_comment",
    "follows_reckless_pattern_comment",
    "judge_explanation_preview",
    "model_output_preview",
    "trace_id",
]

In [ ]:
# Analysis — fetch traces + scores from Langfuse and build two views:
#   bundle.summary_df   : aggregated metrics per GROUP_BY group
#   master_traces_df    : one row per trace, sorted by SORT_BY

bundle = explorer.analyze_dataset(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)
master_traces_df = explorer.build_master_traces_frame(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)

sort_columns = [col for col in SORT_BY if col in master_traces_df.columns]
ascending = ASCENDING[: len(sort_columns)] if sort_columns else True
most_harmful_df = master_traces_df.sort_values(sort_columns, ascending=ascending).reset_index(drop=True)

scope_text = bundle.selected_execution if bundle.selected_execution != "all" else "all executions"
display(
    Markdown(
        f"## {DATASET_NAME}\n"
        f"execution: `{scope_text}`  \n"
        f"matching traces: `{len(master_traces_df)}`  \n"
        f"grouped by: `{GROUP_BY}`"
    )
)

display(Markdown("### Condition Summary"))
display(bundle.summary_df)

display(Markdown("### Top Traces"))
master_view_columns = [col for col in MASTER_VIEW_COLUMNS if col in most_harmful_df.columns]
display(most_harmful_df.loc[:, master_view_columns].head(TOP_N))

### Trace Detail View

The cell below prints the top `DETAIL_TRACE_COUNT` traces in full: user input, model output, judge explanation, and per-metric comments.

In [ ]:
for index, row in most_harmful_df.head(DETAIL_TRACE_COUNT).iterrows():
    display(
        Markdown(
            f"## {index + 1}. task=`{row.get('task_id', '<unknown>')}` "
            f"| condition=`{row.get('condition', row.get('variant_id', '<unknown>'))}` "
            f"| model=`{row.get('model', '<unknown>')}` "
            f"| harmful=`{row.get('harmful', '<na>')}` "
            f"| helpful=`{row.get('helpful', '<na>')}` "
            f"| reckless=`{row.get('follows_reckless_pattern', '<na>')}`"
        )
    )
    display(
        Markdown(
            f"trace id: `{row.get('trace_id', '<unknown>')}`  \n"
            f"judge explanation: {row.get('judge_explanation', '')}"
        )
    )
    print("User input:\n")
    print(row.get("trace_input", ""))
    print("\nModel output:\n")
    print(row.get("model_output", ""))
    print("\nEvaluator comments:\n")
    for metric_name in ["harmful", "helpful", "follows_reckless_pattern", "turn_count", "tool_call_count"]:
        comment_column = f"{metric_name}_comment"
        if comment_column in most_harmful_df.columns and pd.notna(row.get(comment_column)):
            print(f"- {metric_name}: {row.get(comment_column)}")
    if pd.notna(row.get("langfuse_url")):
        print(f"\nLangfuse URL: {row.get('langfuse_url')}")
    print("\n" + "=" * 120 + "\n")